<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 6 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">保留输入、分流拒收、自动验收</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">保留每条原始输入，将合格订单与拒收记录分别保存。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

完成后，你将把十三行原始输入分为十行合格订单与三行拒收记录，核对拒收原因，并用故意注入的错误检验质量规则。请按顺序运行。

[讲义](course6_data_quality_and_schema_validation.md) · [课程入口](../README.md)


## 实验范围

先完成 D05，确认 wwi_customers 存在；本节读取该历史客户表，不修改它。

重建 orders_raw、customers、orders_classified 视图、orders_clean 和 orders_rejected。13 行输入包括三种不同的异常，先全部暂存，再做业务准入。D07 读取本 Lab 的合格输出。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized, expected_failure
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. 保留模拟新订单的原始字段

将金额、订单号和客户号先作为字符串暂存，并使用 input_id 标识每条输入。十三行中有十笔正常订单，以及非法金额、缺失订单号、无效客户各一行。

客户维度读取 D05 的 wwi_customers（663 行），为客户引用检查提供独立依据。这样即使某条输入无法转换类型，也能保留原文并找到原因。


In [ ]:
lab.execute("DROP VIEW IF EXISTS orders_classified")
lab.execute("DROP TABLE IF EXISTS orders_raw")
fields = ", ".join(col + " VARCHAR(100) NULL" for col in ORDER_COLUMNS)
lab.execute(f'CREATE TABLE orders_raw (input_id BIGINT NOT NULL, {fields}) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
raw = fixture("raw_orders.json")
lab.insert("orders_raw", ["input_id", *ORDER_COLUMNS],
           [(r["input_id"], *[None if r[col] is None else str(r[col]) for col in ORDER_COLUMNS]) for r in raw])
expect(lab.query("SELECT COUNT(*) FROM orders_raw"), [(13,)])

lab.execute("DROP TABLE IF EXISTS customers")
lab.execute('CREATE TABLE customers (customer_id BIGINT NOT NULL, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.execute("INSERT INTO customers SELECT CustomerID, CustomerName FROM wwi_customers")
expect(lab.query("SELECT COUNT(*) FROM customers"), [(663,)])


## 2. 按固定规则分流

分类视图通过 TRY_CAST 尝试转换字段，依次检查订单号、金额、客户引用和来源，记录首个命中的拒收原因。随后将合格行写入 orders_clean，将有问题的 input_id 与原因写入 orders_rejected。

预期十行合格、三行拒收；通过 input_id 关联 orders_raw，可以查看错误原文。


In [ ]:
lab.execute("""
CREATE VIEW orders_classified AS
SELECT *, CASE
    WHEN TRY_CAST(order_id AS BIGINT) IS NULL THEN 'INVALID_ORDER_ID'
    WHEN TRY_CAST(order_amount AS DECIMAL(12,2)) IS NULL
      OR TRY_CAST(order_amount AS DECIMAL(12,2)) < 0 THEN 'INVALID_AMOUNT'
    WHEN TRY_CAST(customer_id AS BIGINT) IS NULL THEN 'INVALID_CUSTOMER'
    WHEN TRY_CAST(customer_id AS BIGINT) NOT IN (SELECT customer_id FROM customers) THEN 'INVALID_CUSTOMER'
    WHEN data_source IS NULL OR data_source <> 'COURSE_SIMULATION' THEN 'INVALID_SOURCE'
    ELSE NULL END AS reject_reason
FROM orders_raw
""")
lab.execute("DROP TABLE IF EXISTS orders_clean")
ddl = order_ddl("orders_clean")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS orders_rejected")
lab.execute('CREATE TABLE orders_rejected (input_id BIGINT, reason VARCHAR(32)) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")
lab.execute("INSERT INTO orders_rejected SELECT input_id, reject_reason FROM orders_classified WHERE reject_reason IS NOT NULL")
lab.sql("SELECT reject_reason, COUNT(*) AS input_rows FROM orders_classified GROUP BY reject_reason ORDER BY reject_reason", title="输入分流结果")
lab.sql("SELECT r.input_id, r.order_id, r.order_amount, r.customer_id, x.reason FROM orders_rejected x JOIN orders_raw r ON x.input_id=r.input_id ORDER BY r.input_id", title="三条拒收记录及原始字段")


## 3. 检查业务质量并注入错误

先检查十三条输入均有去向，再核对订单唯一性、客户引用、金额、状态和完整记录。事件时间以本批样本的固定截止时刻为准。

接着追加一笔重复订单，观察检查按预期发现错误；从分类输入重建合格表后，再运行检查，应恢复为十笔合格订单、金额 1400.00。


In [ ]:
expected_rows = order_rows(fixture("orders.json"))
def quality_report():
    expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")])
    expect(lab.query("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders_clean"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE event_time > '2026-01-02 12:00:00' OR event_time IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE status <> 'CREATED' OR paid_amount <> 0 OR refund_amount <> 0"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean WHERE data_source <> 'COURSE_SIMULATION'"), [(0,)])
    # DATE_FORMAT makes the transport representation explicit for fixture comparison.
    projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
    expect(lab.query(f"SELECT {projection} FROM orders_clean ORDER BY order_id"), expected_rows)

expect(lab.query("SELECT input_id, reason FROM orders_rejected ORDER BY input_id"),
       [(11,"INVALID_AMOUNT"),(12,"INVALID_ORDER_ID"),(13,"INVALID_CUSTOMER")])
expect(lab.query("SELECT COUNT(*) FROM orders_classified WHERE reject_reason IS NULL"), [(10,)])
expect(lab.query("SELECT COUNT(*) FROM orders_raw r LEFT JOIN orders_rejected x ON r.input_id=x.input_id WHERE x.input_id IS NULL"), [(10,)])
quality_report()

# Inject one duplicate: the validator must detect it, not just print a warning.
lab.insert("orders_clean", ORDER_COLUMNS, [expected_rows[0]])
with expected_failure("重复订单检查", "已识别重复订单，质量规则生效"):
    quality_report()
lab.execute("TRUNCATE TABLE orders_clean")
lab.execute(f"INSERT INTO orders_clean ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM orders_classified WHERE reject_reason IS NULL")
quality_report()
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_clean", title="恢复后的合格订单")


## 完成与自己动手

核对十三行输入、十行合格、三行拒收，并通过 input_id 查出三种拒收原因。说明客户号 999999 为什么能够转换成整数，却仍会被拒收。

D07 将读取 orders_clean，继续处理这十笔合格模拟订单的状态变化。


## 独立练习

新增规则：金额超过 200 的合格订单进入人工复核，其余接受。使用 CASE 编写一条 SELECT，按 ACCEPT、REVIEW 分组统计订单数与金额。预期 ACCEPT 为 8 笔、850.00，REVIEW 为 2 笔、550.00。先试算新规则，保持 orders_clean 不变供 D07 使用。

在下一格编写并运行代码，完成后再展开参考解答。空白练习不会被自动判定为完成。


In [ ]:
# 在这里编写你的 SQL 或导入请求。


<details>
<summary>参考解答（完成后再展开）</summary>

```python
query = """SELECT CASE WHEN order_amount > 200 THEN 'REVIEW' ELSE 'ACCEPT' END AS decision,
COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_clean GROUP BY decision ORDER BY decision"""
lab.sql(query, title="新增金额复核规则")
expect(lab.query(query), [("ACCEPT",8,"850.00"),("REVIEW",2,"550.00")])
# 只试算新规则，保留 D07 使用的十笔合格订单。
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean"), [(10,"1400.00")])
```

</details>
